# Milestone 3: Correctness check

A program that compares the GPU output with the CPU output, value by value, and prints the largest difference. We'll rerun it after every optimisation in later milestones: a faster kernel is useless if it's wrong.

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project
!ls output

## Step 2: Write the comparison program
It's plain C++ (no GPU needed to compare two files). It reports:
- the **largest difference** between any two matching values (0 to 255)
- **where** that difference first occurs (row, column, colour), to help find bugs
- **how many** values differ by 0, by 1, by 2 and so on
- **PASS** if the largest difference is at most 1, and **FAIL** otherwise

In [ ]:
%%writefile compare_outputs.cpp
// compare_outputs.cpp
// Compares two 8-bit RGB output files value by value.
// Usage: compare_outputs referenceFile testFile
// Example: compare_outputs output/cpu_output.bin output/gpu_naive_output.bin
// We will reuse this for every GPU version in later milestones.

#include <cstdlib>
#include <fstream>
#include <iostream>
#include <vector>

#include "isp_io.h"

// Reads a whole binary file into a byte vector. Returns false on failure.
static bool readWholeFile(const char* filePath, std::vector<unsigned char>& fileBytes) {
    // std::ios::ate opens the file with the read position at the END,
    // so tellg() tells us the file size straight away.
    std::ifstream inputFile(filePath, std::ios::binary | std::ios::ate);
    if (!inputFile.is_open()) {
        std::cout << "ERROR: could not open " << filePath << std::endl;
        return false;
    }
    long long fileSize = static_cast<long long>(inputFile.tellg());
    inputFile.seekg(0);

    fileBytes.resize(fileSize);
    char* destinationBytes = reinterpret_cast<char*>(fileBytes.data());
    inputFile.read(destinationBytes, fileSize);
    return true;
}

// argc is how many words were typed on the command line (including the
// program name), and argv holds those words. argv[1] and argv[2] are our files.
int main(int argc, char* argv[]) {
    if (argc != 3) {
        std::cout << "Usage: compare_outputs referenceFile testFile" << std::endl;
        return 1;
    }

    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int imageWidth = metadata.imageWidth;

    std::vector<unsigned char> referenceBytes;
    std::vector<unsigned char> testBytes;
    bool referenceOk = readWholeFile(argv[1], referenceBytes);
    bool testOk = readWholeFile(argv[2], testBytes);
    if (!referenceOk || !testOk) {
        return 1;
    }
    if (referenceBytes.size() != testBytes.size()) {
        std::cout << "ERROR: files have different sizes" << std::endl;
        return 1;
    }

    // Walk through every value. Record the largest difference, where it was,
    // and how many values differ by each amount (0, 1, 2, ... up to 255).
    long long valueCount = static_cast<long long>(referenceBytes.size());
    std::vector<long long> countPerDifference(256, 0);
    int largestDifference = 0;
    long long largestDifferenceIndex = 0;

    for (long long valueIndex = 0; valueIndex < valueCount; valueIndex++) {
        int referenceValue = referenceBytes[valueIndex];
        int testValue = testBytes[valueIndex];
        int difference = std::abs(referenceValue - testValue);

        countPerDifference[difference] = countPerDifference[difference] + 1;

        if (difference > largestDifference) {
            largestDifference = difference;
            largestDifferenceIndex = valueIndex;
        }
    }

    // Turn the index back into (row, column, channel) so a problem can be located.
    long long pixelIndex = largestDifferenceIndex / 3;
    int channel = static_cast<int>(largestDifferenceIndex % 3);
    long long row = pixelIndex / imageWidth;
    long long column = pixelIndex % imageWidth;

    std::cout << "Values compared      : " << valueCount << std::endl;
    std::cout << "Largest difference   : " << largestDifference << " (out of 255)" << std::endl;
    if (largestDifference > 0) {
        std::cout << "  first seen at row " << row << ", column " << column
                  << ", channel " << channel << " (0=red 1=green 2=blue)" << std::endl;
    }

    std::cout << "How many values differ by each amount:" << std::endl;
    for (int difference = 0; difference <= largestDifference; difference++) {
        double percent = 100.0 * countPerDifference[difference] / valueCount;
        std::cout << "  differ by " << difference << ": " << countPerDifference[difference]
                  << " values (" << percent << " %)" << std::endl;
    }

    // A difference of 1 can come from harmless floating-point rounding (see the
    // notebook). Anything bigger means the two programs do different maths.
    if (largestDifference <= 1) {
        std::cout << "RESULT: PASS" << std::endl;
        return 0;
    } else {
        std::cout << "RESULT: FAIL - look for a bug near the location above" << std::endl;
        return 1;
    }
}


In [ ]:
!g++ -O2 -std=c++17 -Wall -Wextra -o /content/compare_outputs compare_outputs.cpp isp_io.cpp

## Step 3: Compare CPU and GPU (CHECKPOINT)
**Expected:** `Values compared : 67291560` (5796 × 3870 pixels × 3 colours), a largest difference of **0 or 1**, almost all values in the "differ by 0" row, and `RESULT: PASS`.

**Paste the full output to me.**

In [ ]:
!/content/compare_outputs output/cpu_output.bin output/gpu_naive_output.bin

## Why a difference of 1 is acceptable

Both programs do the same maths, but **floating-point numbers are not exact**. A `float` keeps only about 7 significant digits, and every operation rounds its result to the nearest value a float can hold. Two programs that do the "same" calculation in a slightly different way can therefore end up with results that differ in the 7th digit. On the CPU and GPU, three things cause this:

1. **Fused multiply-add (FMA).** In the denoise stage we compute `total = total + value * weight`. By default, nvcc turns this into a single GPU instruction that multiplies and adds with **one** rounding at the end. The CPU (with our g++ settings) multiplies, rounds, then adds and rounds again. The GPU answer is actually slightly *more* accurate, but it's different.
2. **`pow` is implemented differently.** The CPU's `std::pow` and the GPU's `powf` are written by different people. Both are accurate to within about one unit in the last digit, but not always the *same* last digit.
3. **Compiler reordering.** A compiler may evaluate a sum in a different order, and `(a + b) + c` doesn't always equal `a + (b + c)` exactly in floating point.

These differences are around 0.0000001. They only become visible at the very end, when we convert 0.0–1.0 to whole numbers 0–255. If a value lands almost exactly halfway between two whole numbers, for example 127.4999999 on one machine and 127.5000001 on the other, one rounds down to 127 and the other rounds up to 128. That's a difference of 1, out of 255. You can't see it in a photo, and it isn't a bug.

**A real bug looks completely different:** differences of 5, 50 or 200, usually clustered in one area such as the image edge (a border-handling mistake) or on every second pixel (a Bayer-pattern mistake). That's why the program reports *where* the largest difference is.